# 03a - Dummy Baseline
Referencni spodni latka pro nevyrovnany dataset (dropout ~24 %). Vysledky se ukladaji do `results/dummy_results.pkl` pro porovnani v `03c

In [1]:
import joblib
import warnings
import os
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

print("Rozlozeni trid v y_train:")
print(y_train.value_counts(normalize=True).to_string())

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


Rozlozeni trid v y_train:
Dropout
0    0.764625
1    0.235375


## Definice dummy klasifikatoru
- **most_frequent** - vzdy predikuje majoritni tridu (0); odhalí, zda je accuracy nasich modelu vubec relevantni
- **stratified** - nahodne predikce zachovavajici rozlozeni trid; ferovoejsi spodni hranice pro ROC-AUC

In [2]:
dummy_models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent', random_state=42),
    'Dummy (stratified)':    DummyClassifier(strategy='stratified',    random_state=42),
}

dummy_results = {}

for name, model in dummy_models.items():
    model.fit(X_train_prep, y_train)

    y_pred = model.predict(X_test_prep)

    # most_frequent nema smysluplne predict_proba pro roc_auc -> fallback na 0.5
    try:
        y_proba = model.predict_proba(X_test_prep)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    except Exception:
        roc_auc = 0.5

    dummy_results[name] = {
        'model':    model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc':  roc_auc,
        'report':   classification_report(y_test, y_pred),
        'y_pred':   y_pred,
    }

    print(f"{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {dummy_results[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {dummy_results[name]['roc_auc']:.4f}")
    print(f"\n{dummy_results[name]['report']}")

  Dummy (most_frequent)
  Accuracy : 0.7645
  ROC-AUC  : 0.5000

              precision    recall  f1-score   support

           0       0.76      1.00      0.87      1529
           1       0.00      0.00      0.00       471

    accuracy                           0.76      2000
   macro avg       0.38      0.50      0.43      2000
weighted avg       0.58      0.76      0.66      2000

  Dummy (stratified)
  Accuracy : 0.6490
  ROC-AUC  : 0.5133

              precision    recall  f1-score   support

           0       0.77      0.77      0.77      1529
           1       0.26      0.26      0.26       471

    accuracy                           0.65      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.65      0.65      0.65      2000



In [3]:
os.makedirs('../results', exist_ok=True)
joblib.dump(dummy_results, '../results/dummy_results.pkl')
print("Dummy vysledky ulozeny do results/dummy_results.pkl")

Dummy vysledky ulozeny do results/dummy_results.pkl
